# CRISP-DM Stage 4: Evaluation

1. **Internal Evaluation Metrics & Visualizations** (Silhouette, DBI, CHI, PCA 2D)
2. **AI Cluster Interpretation & Policy Recommendations** (Cloudflare Workers AI)

In [ ]:
# Parameter Injeksi (DVC / Papermill)
target_silhouette_min = 0.50
model = "@cf/openai/gpt-oss-120b"
max_tokens = 3000
selected_features = [
    'total_koperasi',
    'rasio_nib',
    'rasio_npwp',
    'rasio_rat',
    'simpanan_pokok',
    'simpanan_wajib',
    'volume_transaksi',
    'nilai_transaksi'
]


## 1. Internal Metrics & Visualizations

In [ ]:
import os, json, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from config import (
    SCALED_FEATURES_CSV, CLUSTERED_REGENCIES_CSV, EVALUATE_METRICS_JSON,
    EVALUATE_REPORT_MD, FIGURES_DIR
)

df_clustered = pd.read_csv(CLUSTERED_REGENCIES_CSV)
df_scaled = pd.read_csv(SCALED_FEATURES_CSV)

scaled_cols = [f"scaled_{c}" for c in selected_features if f"scaled_{c}" in df_scaled.columns]
X_scaled = df_scaled[scaled_cols].values if scaled_cols else df_scaled.values
labels = df_clustered['cluster_label'].values

sil_score = round(float(silhouette_score(X_scaled, labels)), 4)
ch_score = round(float(calinski_harabasz_score(X_scaled, labels)), 2)
db_score = round(float(davies_bouldin_score(X_scaled, labels)), 4)
num_clusters = len(np.unique(labels))

# Simpan Metrik
metrics = {
    "clustering_metrics": {
        "number_of_clusters": num_clusters, "silhouette_score": sil_score,
        "calinski_harabasz_score": ch_score, "davies_bouldin_score": db_score,
        "target_silhouette_achieved": bool(sil_score >= target_silhouette_min)
    }
}
os.makedirs(os.path.dirname(EVALUATE_METRICS_JSON), exist_ok=True)
with open(EVALUATE_METRICS_JSON, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

# PCA 2D Plot
os.makedirs(FIGURES_DIR, exist_ok=True)
X_pca = PCA(n_components=2).fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PCA1', 'PCA2'])
df_pca['cluster'] = [f"Klaster {lbl}" for lbl in labels]

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='cluster', data=df_pca, palette='tab10', s=60, alpha=0.85)
plt.title('Proyeksi 2D Klasterisasi (Principal Component Analysis)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eval_pca_projection.png'))
plt.show()

# Cluster Distribution Barplot
cluster_counts = df_clustered['cluster_label'].value_counts().sort_index()
plt.figure(figsize=(8, 4.5))
sns.barplot(x=cluster_counts.index.map(lambda x: f"Klaster {x}"), y=cluster_counts.values, hue=cluster_counts.index, legend=False, palette='Blues_r')
plt.title('Distribusi Jumlah Anggota Kabupaten/Kota per Klaster')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eval_cluster_distribution.png'))
plt.show()

# Profiling Klaster Table via pandas to_markdown
active_features = [c for c in selected_features if c in df_clustered.columns]
profile_df = df_clustered.groupby('cluster_label')[active_features].mean().round(2)
profile_table = profile_df.to_markdown()

dist_list = "".join([f"- **Klaster {cid}**: {count} Kabupaten/Kota ({round(count/len(df_clustered)*100, 2)}%)\n" for cid, count in cluster_counts.items()])

eval_report_text = f"""# Laporan Evaluasi Klasterisasi (CRISP-DM Evaluation)

## Metrik Validasi Internal Klaster
- **Jumlah Klaster (K)**: {num_clusters}
- **Silhouette Coefficient**: {sil_score} (Target: >={target_silhouette_min})
- **Calinski-Harabasz Index**: {ch_score}
- **Davies-Bouldin Index**: {db_score}

## Distribusi Anggota Klaster
{dist_list}

## Profil Rata-Rata Karakteristik per Klaster
{profile_table}
"""
with open(EVALUATE_REPORT_MD, 'w', encoding='utf-8') as f:
    f.write(eval_report_text)

display(Markdown(eval_report_text))


## 2. AI Cluster Interpretation

In [ ]:
import re, urllib.request
from utils.env_utils import load_env
from config import INTERPRET_REPORT_MD, INTERPRET_LABELS_JSON

load_env()
account_id, api_token = os.environ.get("CF_ACCOUNT_ID"), os.environ.get("CF_API_TOKEN")

profile_text = ""
for label, group in df_clustered.groupby('cluster_label'):
    profile_text += f"- **Klaster {label}** ({len(group)} Kab/Kota):\n"
    for col in active_features:
        m = group[col].mean()
        lbl = col.replace('_', ' ').title()
        profile_text += f"  - Rata-rata {lbl}: Rp {m:,.2f}\n" if "nilai" in col or "simpanan" in col else f"  - Rata-rata {lbl}: {m:.2f}%\n" if "rasio" in col else f"  - Rata-rata {lbl}: {m:,.2f}\n"

prompt = f"""Anda adalah pakar analis data koperasi Indonesia.
Analisis hasil pengelompokan K-Means KDMP:
{profile_text}
Keluarkan format JSON murni:
{{
  "labels": {{
    "0": {{"label_name": "Klaster 0 - [Nama Tipologi]", "description": "Deskripsi..."}},
    "1": {{"label_name": "Klaster 1 - [Nama Tipologi]", "description": "Deskripsi..."}}
  }},
  "report": "# Laporan Interpretasi AI Klaster Koperasi Wilayah\\n\\n[Tuliskan analisis komprehensif 2-3 paragraf.]"
}}
PENTING: Output HANYA JSON valid."""

if account_id and api_token:
    url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/ai/run/{model}"
    headers = {"Authorization": f"Bearer {api_token}", "Content-Type": "application/json"}
    payload = {"messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens}
    req = urllib.request.Request(url, data=json.dumps(payload).encode('utf-8'), headers=headers, method='POST')
    try:
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode('utf-8'))
            res_obj = data.get("result", {})
            ai_text = (res_obj.get("choices", [{}])[0].get("message", {}).get("content") or res_obj.get("response") or "").strip()
            clean_json = re.sub(r"^```(?:json)?\n|\n```$", "", ai_text).strip()
            s_idx, e_idx = clean_json.find('{'), clean_json.rfind('}')
            parsed = json.loads(clean_json[s_idx:e_idx+1])
            
            with open(INTERPRET_LABELS_JSON, 'w', encoding='utf-8') as f:
                json.dump(parsed.get("labels", {}), f, indent=2)
            
            ai_rep_content = parsed.get("report", "")
            with open(INTERPRET_REPORT_MD, 'w', encoding='utf-8') as f:
                f.write(ai_rep_content)
            display(Markdown(ai_rep_content))
    except Exception as e:
        print(f"Gagal memanggil Workers AI: {e}")
else:
    print("CF credentials tidak ditemukan.")
